# PER vs Energy Bubble Plot Template

Fill in the nested dictionary below with three sizes per model family and five seeds per size. The notebook averages PER and energy across seeds, then plots each averaged model-size point on a square bubble chart where bubble area represents parameter count.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.lines import Line2D
from matplotlib.ticker import PercentFormatter

mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42

GRID_COLOR = '#d9e0e7'
BACKGROUND = '#fbfcfd'
EDGE_COLOR = '#173042'

MODEL_COLORS = {
    'LSTM': '#4c78a8',
    'MLP': '#f58518',
    'SOTA GRU': '#54a24b',
    'SNN': '#b54a4a',
}


In [ ]:
# Replace placeholder values below with your measured numbers.
# `parameters` should be the raw parameter count, not millions.
# Each `per_seeds` and `energy_uJ_seeds` entry should contain exactly 5 seed values.

model_stats = {
    'LSTM': {
        'Small': {
            'parameters': np.nan,
            'per_seeds': [np.nan, np.nan, np.nan, np.nan, np.nan],
            'energy_uJ_seeds': [np.nan, np.nan, np.nan, np.nan, np.nan],
        },
        'Medium': {
            'parameters': 4479529,
            'per_seeds': [0.21772, 0.21555, np.nan, np.nan, np.nan],
            'energy_uJ_seeds': [np.nan, np.nan, np.nan, np.nan, np.nan],
        },
        'Large': {
            'parameters': 7309097,
            'per_seeds': [0.20729, 0.20668, 0.20970, 0.20731, 0.20869],
            'energy_uJ_seeds': [np.nan, np.nan, np.nan, np.nan, np.nan],
        },
    },
    'MLP': {
        'Small': {
            'parameters': np.nan,
            'per_seeds': [np.nan, np.nan, np.nan, np.nan, np.nan],
            'energy_uJ_seeds': [np.nan, np.nan, np.nan, np.nan, np.nan],
        },
        'Medium': {
            'parameters': np.nan,
            'per_seeds': [np.nan, np.nan, np.nan, np.nan, np.nan],
            'energy_uJ_seeds': [np.nan, np.nan, np.nan, np.nan, np.nan],
        },
        'Large': {
            'parameters': np.nan,
            'per_seeds': [np.nan, np.nan, np.nan, np.nan, np.nan],
            'energy_uJ_seeds': [np.nan, np.nan, np.nan, np.nan, np.nan],
        },
    },
    'SOTA GRU': {
        'Small': {
            'parameters': np.nan,
            'per_seeds': [np.nan, np.nan, np.nan, np.nan, np.nan],
            'energy_uJ_seeds': [np.nan, np.nan, np.nan, np.nan, np.nan],
        },
        'Medium': {
            'parameters': np.nan,
            'per_seeds': [np.nan, np.nan, np.nan, np.nan, np.nan],
            'energy_uJ_seeds': [np.nan, np.nan, np.nan, np.nan, np.nan],
        },
        'Large': {
            'parameters': np.nan,
            'per_seeds': [np.nan, np.nan, np.nan, np.nan, np.nan],
            'energy_uJ_seeds': [np.nan, np.nan, np.nan, np.nan, np.nan],
        },
    },
    'SNN': {
        'Small': {
            'parameters': np.nan,
            'per_seeds': [np.nan, np.nan, np.nan, np.nan, np.nan],
            'energy_uJ_seeds': [np.nan, np.nan, np.nan, np.nan, np.nan],
        },
        'Medium': {
            'parameters': np.nan,
            'per_seeds': [np.nan, np.nan, np.nan, np.nan, np.nan],
            'energy_uJ_seeds': [np.nan, np.nan, np.nan, np.nan, np.nan],
        },
        'Large': {
            'parameters': np.nan,
            'per_seeds': [np.nan, np.nan, np.nan, np.nan, np.nan],
            'energy_uJ_seeds': [np.nan, np.nan, np.nan, np.nan, np.nan],
        },
    },
}

save_path = 'per_energy_model_size_bubble.png'
figure_size = (7.0, 7.0)


In [ ]:
def format_parameter_count(value):
    if value >= 1e9:
        return f'{value / 1e9:.2f}B'
    if value >= 1e6:
        return f'{value / 1e6:.2f}M'
    if value >= 1e3:
        return f'{value / 1e3:.0f}K'
    return f'{value:.0f}'


def bubble_area_from_parameters(parameters, min_area=240, max_area=1800):
    parameters = np.asarray(parameters, dtype=float)
    if parameters.size == 1 or np.allclose(parameters.min(), parameters.max()):
        return np.full(parameters.shape, (min_area + max_area) / 2.0)
    scale = (parameters - parameters.min()) / (parameters.max() - parameters.min())
    return min_area + scale * (max_area - min_area)


records = []
for model_name, size_dict in model_stats.items():
    for size_name, metrics in size_dict.items():
        per_seeds = np.asarray(metrics.get('per_seeds', []), dtype=float)
        energy_seeds = np.asarray(metrics.get('energy_uJ_seeds', []), dtype=float)
        parameters = float(metrics.get('parameters', np.nan))

        if per_seeds.size != 5 or energy_seeds.size != 5:
            raise ValueError(
                f'{model_name} {size_name} must contain exactly 5 values for both per_seeds and energy_uJ_seeds.'
            )

        if np.isnan(parameters) or np.isnan(per_seeds).any() or np.isnan(energy_seeds).any():
            continue

        records.append(
            {
                'model_name': model_name,
                'size_name': size_name,
                'label': f'{model_name} {size_name}',
                'parameters': parameters,
                'per_mean': float(np.mean(per_seeds)),
                'per_std': float(np.std(per_seeds, ddof=1)),
                'energy_mean': float(np.mean(energy_seeds)),
                'energy_std': float(np.std(energy_seeds, ddof=1)),
                'color': MODEL_COLORS.get(model_name, '#4c78a8'),
            }
        )

if not records:
    raise ValueError('Fill in at least one complete model-size entry before plotting.')

records = sorted(records, key=lambda record: (record['model_name'], record['parameters']))

labels = [record['label'] for record in records]
parameters = np.array([record['parameters'] for record in records], dtype=float)
per_means = np.array([record['per_mean'] for record in records], dtype=float)
per_stds = np.array([record['per_std'] for record in records], dtype=float)
energy_means = np.array([record['energy_mean'] for record in records], dtype=float)
energy_stds = np.array([record['energy_std'] for record in records], dtype=float)
colors = [record['color'] for record in records]
bubble_sizes = bubble_area_from_parameters(parameters)

print('Model-size averages included in plot:')
for record in records:
    print(
        f"  {record['label']}: params={format_parameter_count(record['parameters'])}, "
        f"PER={record['per_mean']:.4f} +/- {record['per_std']:.4f}, "
        f"energy={record['energy_mean']:.3f} +/- {record['energy_std']:.3f} uJ"
    )


In [ ]:
fig, ax = plt.subplots(figsize=figure_size, constrained_layout=True)
fig.patch.set_facecolor('white')
ax.set_facecolor(BACKGROUND)

ax.errorbar(
    energy_means,
    per_means,
    xerr=energy_stds,
    yerr=per_stds,
    fmt='none',
    ecolor='#93a1af',
    elinewidth=1.0,
    capsize=3,
    alpha=0.95,
    zorder=1,
)

for record, bubble_size in zip(records, bubble_sizes):
    ax.scatter(
        record['energy_mean'],
        record['per_mean'],
        s=bubble_size,
        c=record['color'],
        edgecolors=EDGE_COLOR,
        linewidths=1.1,
        alpha=0.92,
        zorder=3,
    )
    ax.annotate(
        record['label'],
        xy=(record['energy_mean'], record['per_mean']),
        xytext=(8, 8),
        textcoords='offset points',
        fontsize=9,
        color=EDGE_COLOR,
    )

ax.set_xlabel('Energy per Inference (uJ)', labelpad=10)
ax.set_ylabel('Phoneme Error Rate (PER)', labelpad=10)
ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=1))

ax.grid(True, color=GRID_COLOR, linewidth=0.8, alpha=0.85)
ax.set_axisbelow(True)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('#7a8793')
ax.spines['bottom'].set_color('#7a8793')
ax.tick_params(colors='#2d3b45')
ax.margins(x=0.12, y=0.12)

family_handles = [
    Line2D(
        [0],
        [0],
        marker='o',
        linestyle='',
        markerfacecolor=color,
        markeredgecolor=EDGE_COLOR,
        markersize=9,
        label=model_name,
    )
    for model_name, color in MODEL_COLORS.items()
]

legend_parameter_values = np.unique(np.quantile(parameters, [0.0, 0.5, 1.0]))
legend_bubble_sizes = bubble_area_from_parameters(legend_parameter_values)
size_handles = [
    Line2D(
        [0],
        [0],
        marker='o',
        linestyle='',
        markerfacecolor='white',
        markeredgecolor='#7a8793',
        markersize=np.sqrt(size) / 1.7,
        label=format_parameter_count(value),
    )
    for value, size in zip(legend_parameter_values, legend_bubble_sizes)
]

family_legend = ax.legend(
    handles=family_handles,
    title='Model family',
    loc='upper right',
    frameon=True,
    facecolor='white',
    edgecolor='#d0d7de',
)
ax.add_artist(family_legend)

ax.legend(
    handles=size_handles,
    title='Parameter count',
    loc='lower right',
    frameon=True,
    facecolor='white',
    edgecolor='#d0d7de',
)

plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=figure_size, constrained_layout=True)
fig.patch.set_facecolor('white')
ax.set_facecolor(BACKGROUND)

ax.errorbar(
    energy_means,
    per_means,
    xerr=energy_stds,
    yerr=per_stds,
    fmt='none',
    ecolor='#93a1af',
    elinewidth=1.0,
    capsize=3,
    alpha=0.95,
    zorder=1,
)

for record, bubble_size in zip(records, bubble_sizes):
    ax.scatter(
        record['energy_mean'],
        record['per_mean'],
        s=bubble_size,
        c=record['color'],
        edgecolors=EDGE_COLOR,
        linewidths=1.1,
        alpha=0.92,
        zorder=3,
    )
    ax.annotate(
        record['label'],
        xy=(record['energy_mean'], record['per_mean']),
        xytext=(8, 8),
        textcoords='offset points',
        fontsize=9,
        color=EDGE_COLOR,
    )

ax.set_xlabel('Energy per Inference (uJ)', labelpad=10)
ax.set_ylabel('Phoneme Error Rate (PER)', labelpad=10)
ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=1))
ax.grid(True, color=GRID_COLOR, linewidth=0.8, alpha=0.85)
ax.set_axisbelow(True)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('#7a8793')
ax.spines['bottom'].set_color('#7a8793')
ax.tick_params(colors='#2d3b45')
ax.margins(x=0.12, y=0.12)

family_handles = [
    Line2D(
        [0],
        [0],
        marker='o',
        linestyle='',
        markerfacecolor=color,
        markeredgecolor=EDGE_COLOR,
        markersize=9,
        label=model_name,
    )
    for model_name, color in MODEL_COLORS.items()
]

legend_parameter_values = np.unique(np.quantile(parameters, [0.0, 0.5, 1.0]))
legend_bubble_sizes = bubble_area_from_parameters(legend_parameter_values)
size_handles = [
    Line2D(
        [0],
        [0],
        marker='o',
        linestyle='',
        markerfacecolor='white',
        markeredgecolor='#7a8793',
        markersize=np.sqrt(size) / 1.7,
        label=format_parameter_count(value),
    )
    for value, size in zip(legend_parameter_values, legend_bubble_sizes)
]

family_legend = ax.legend(
    handles=family_handles,
    title='Model family',
    loc='upper right',
    frameon=True,
    facecolor='white',
    edgecolor='#d0d7de',
)
ax.add_artist(family_legend)

ax.legend(
    handles=size_handles,
    title='Parameter count',
    loc='lower right',
    frameon=True,
    facecolor='white',
    edgecolor='#d0d7de',
)

fig.savefig(save_path, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved figure to: {save_path}')
